In [ ]:
#This script takes two .txt Bible files and the plg_verses_new.csv as input and returns aligned ENG+LANG files with all matching verse numbers

In [ ]:
from pathlib import Path

def load_bible_txt_as_dict(path: str | Path) -> dict[str, str]:
    """
    Parse a monolingual Bible .txt file:
    - Skips metadata lines starting with '#'
    - Expects: '<VERSE_ID><whitespace><TEXT...>'
    Returns a dict mapping verse_id -> text.
    """
    path = Path(path)
    verses: dict[str, str] = {}

    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            # Split once: supports tab or space delimiters
            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                continue  # why: ignore malformed lines quietly
            vid, text = parts[0], parts[1].strip()
            if vid in verses:
                # why: last one wins; avoids duplication noise if file has repeats
                pass
            verses[vid] = text

    return verses

# Example usage — set your file paths
eng_path = "eng-x-bible-newsimplified.txt"      # e.g., ENG file
other_path = "cmn-x-bible-newsimplified.txt"     # e.g., DEU file

eng_map = load_bible_txt_as_dict(eng_path)
other_map = load_bible_txt_as_dict(other_path)

print(f"ENG verses:  {len(eng_map)}")
print(f"OTHER verses:{len(other_map)}")
# Quick spot-check:
for k in list(eng_map.keys())[:5]:
    print(k, "→", eng_map[k][:60], "...")

In [ ]:
import pandas as pd
from pathlib import Path
import csv

# === Step 5: Build bilingual CSV of matching perfect-like verses ===

# Input files
plg_path = Path("plg_verses_new.csv")  # from Step 3
eng_path = Path("eng-x-bible-newsimplified.txt")
other_path = Path("cmn-x-bible-newsimplified.txt")

# Language codes
lang1 = "ENG"
lang2 = "CMN"

# --- Load helper from previous cell (should already be defined) ---
# load_bible_txt_as_dict()

eng_map = load_bible_txt_as_dict(eng_path)
other_map = load_bible_txt_as_dict(other_path)

# --- Load list of verse IDs with perfect-like constructions ---
plg_ids = set(pd.read_csv(plg_path)["verse_id"].astype(str))

print(f"Loaded {len(plg_ids)} verse IDs from {plg_path.name}")

# --- Keep only verse IDs that appear in BOTH files ---
common_ids = [vid for vid in plg_ids if vid in eng_map and vid in other_map]
print(f"✅ {len(common_ids)} verses found in both ENG and {lang2}")

# --- Build DataFrame ---
records = []
for vid in common_ids:
    records.append({"verse_id": vid, "language": lang1, "text": eng_map[vid]})
    records.append({"verse_id": vid, "language": lang2, "text": other_map[vid]})

df_out = pd.DataFrame(records).sort_values(["verse_id", "language"]).reset_index(drop=True)

# --- Save output ---
output_path = Path(f"bible_{lang1.lower()}_{lang2.lower()}_perfects.csv")
df_out.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")

print(f"✅ Done! Created bilingual CSV: {output_path}")
print(f"Rows: {len(df_out)}  (≈ {len(common_ids)} verses × 2)")
display(df_out.head(6))

In [2]:
from pathlib import Path

def load_bible_txt_as_dict(path: str | Path) -> dict[str, str]:
    """
    Parse a monolingual Bible .txt file:
    - Skips metadata lines starting with '#'
    - Expects: '<VERSE_ID><whitespace><TEXT...>'
    Returns a dict mapping verse_id -> text.
    """
    path = Path(path)
    verses: dict[str, str] = {}

    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            # Split once: supports tab or space delimiters
            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                continue  # why: ignore malformed lines quietly
            vid, text = parts[0], parts[1].strip()
            if vid in verses:
                # why: last one wins; avoids duplication noise if file has repeats
                pass
            verses[vid] = text

    return verses

In [3]:
import pandas as pd
from pathlib import Path
import csv

# === Step 6: Build multilingual CSV of matching PLG verses ===

# Input files
plg_path = Path("plg_verses_new.csv")  # the list of PLG verse IDs
bible_files = {
    "ENG": Path("eng_bib.txt"),
    "DEU": Path("deu_bib.txt"),
    "CMN": Path("cmn_bib.txt"),
    "JPN": Path("jpn_bib.txt"),
}

# --- Load all Bibles into dictionaries ---
bibles = {lang: load_bible_txt_as_dict(path) for lang, path in bible_files.items()}
for lang, verses in bibles.items():
    print(f"{lang}: {len(verses)} verses loaded")

# --- Load list of verse IDs with PLGs ---
plg_ids = set(pd.read_csv(plg_path)["verse_id"].astype(str))
print(f"PLG verse IDs: {len(plg_ids)}")

# --- Keep only verses present in *all four* files ---
common_ids = plg_ids.copy()
for lang, verses in bibles.items():
    common_ids &= set(verses.keys())

print(f"✅ {len(common_ids)} verses found in ALL {len(bibles)} languages")

# --- Build DataFrame ---
records = []
for vid in sorted(common_ids):
    for lang, verses in bibles.items():
        records.append({
            "verse_id": vid,
            "lang": lang,
            "text": verses[vid],
        })

df_out = pd.DataFrame(records)

# --- Save output ---
output_path = Path("bible_multilingual_plg_test.csv")
df_out.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8")

print(f"✅ Done! Created multilingual CSV: {output_path}")
print(f"Rows: {len(df_out)}  (≈ {len(common_ids)} verses × {len(bibles)})")
display(df_out.head(12))

ENG: 31035 verses loaded
DEU: 31157 verses loaded
CMN: 31074 verses loaded
JPN: 31105 verses loaded
PLG verse IDs: 4453
✅ 3353 verses found in ALL 4 languages
✅ Done! Created multilingual CSV: bible_multilingual_plg_test.csv
Rows: 13412  (≈ 3353 verses × 4)


,verse_id,lang,text
0,10001010,ENG,» I went up to him and killed him . I knew tha...
1,10001010,DEU,Da trat ich auf ihn zu und gab ihm den Todesst...
2,10001010,CMN,我就站到他身边去 ， 把他杀了 ， 因为我知道他 一 倒下 ， 就不能再活了。我摘下他头上的...
3,10001010,JPN,そこで、わたしはそのそばにいって彼を殺しました。彼がすでに倒れて、生きることのできないのを知...
4,10001012,ENG,They grieved and mourned and fasted until even...
5,10001012,DEU,und sie hielten die Totenklage und weinten und...
6,10001012,CMN,他们为了扫罗、他的儿子约拿单、耶和华的子民和以色列家的缘故悲哀、哭泣、禁食直到晚上 ， 因为...
7,10001012,JPN,彼らはサウルのため、またその子ヨナタンのため、また主の民のため、またイスラエルの家のために悲...
8,10001016,ENG,David said to him : » Your blood is on your he...
9,10001016,DEU,David aber sprach zu ihm : Dein Blut über dein...


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

def has_have_vbn(text: str) -> bool:
    doc = nlp(text)
    for i, tok in enumerate(doc):
        if tok.pos_ == "AUX" and tok.lemma_ == "have":
            for j in range(i + 1, len(doc)):
                if doc[j].pos_ in {"VERB", "AUX"} and doc[j].tag_ == "VBN":
                    return True
            return False
    return False

# New Testament IDs in your ENG Bible (books 40–66)
eng_nt_ids = [vid for vid in bibles["ENG"].keys() if int(vid) >= 40000000]

# Count verses that match the rule
eng_nt_have_vbn = sum(has_have_vbn(bibles["ENG"][vid]) for vid in eng_nt_ids)
eng_nt_have_vbn

In [ ]:
[len(set(load_bible_txt_as_dict(p).keys())) for p in bible_files.values()]

In [ ]:
[len(plg_ids & set(v.keys())) for v in bibles.values()]

In [ ]:
import pandas as pd
df_plg = pd.read_csv("plg_verses_new.csv")
df_plg.head(), df_plg.columns.tolist(), len(df_plg)

In [ ]:
for lang, verses in bibles.items():
    overlap = len(set(df_plg["verse_id"].astype(str)) & set(verses.keys()))
    print(f"{lang}: {overlap}")

In [ ]:
list(bibles["ENG"].keys())[:10]

In [ ]:
(df_plg["verse_id"].astype(int) >= 40000000).sum()